# RNN Text Classification — Cellula Assignment

This notebook trains a deep-learning sequence classification model using PyTorch.

**Important:** Every important line is commented/explained so you can follow what the code does. The code automatically uses CUDA when a CUDA-enabled PyTorch installation and GPU are available.

## 1. Import the libraries

We import the libraries needed for data handling, text preprocessing, model training, evaluation, plots, and the confusion matrix.

In [ ]:
import os  # Used for working with file and folder paths.
import re  # Used for cleaning text with regular expressions.
import random  # Used to make the random operations reproducible.
import numpy as np  # Used for numerical operations.
import pandas as pd  # Used to load and manipulate the CSV dataset.
import matplotlib.pyplot as plt  # Used to create training graphs.

import torch  # Main PyTorch library.
import torch.nn as nn  # Contains neural-network layers such as RNN and Linear.
from torch.utils.data import Dataset, DataLoader  # Used to build batches of training data.

from sklearn.model_selection import train_test_split  # Splits the dataset into train/validation/test sets.
from sklearn.preprocessing import LabelEncoder  # Converts text labels into numbers.
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay  # Evaluation tools.

SEED = 42  # A fixed seed makes the experiment more reproducible.
random.seed(SEED)  # Fix Python's random generator.
np.random.seed(SEED)  # Fix NumPy's random generator.
torch.manual_seed(SEED)  # Fix PyTorch's CPU random generator.
if torch.cuda.is_available():  # Check whether CUDA/GPU is available.
    torch.cuda.manual_seed_all(SEED)  # Fix PyTorch's CUDA random generator.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Select GPU if available; otherwise use CPU.
print('Using device:', device)  # Show whether the notebook is using CUDA or CPU.


In [ ]:
import sys
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)

test_tensor = torch.tensor([1, 2, 3])
print("Tensor:", test_tensor)

try:
    print("NumPy conversion:", test_tensor.numpy())
except Exception as e:
    print("ERROR:", e)

## 2. Load the dataset

The notebook first looks for `cellula toxic data.csv` in the current folder. If it is not there, it also checks the parent folder.

In [ ]:
DATA_FILE = 'cellula toxic data.csv'  # Name of the CSV dataset.

if not os.path.exists(DATA_FILE):  # Check whether the CSV is in the notebook's current folder.
    DATA_FILE = os.path.join('..', 'cellula toxic data.csv')  # Otherwise check the parent folder.

df = pd.read_csv(DATA_FILE)  # Read the CSV file into a pandas DataFrame.
print('Dataset shape:', df.shape)  # Print the number of rows and columns.
print(df.head())  # Display the first five rows so we can inspect the data.
print('\nColumn names:', list(df.columns))  # Display all column names.


In [ ]:
# Find the query/text column by checking common names used by text + image datasets.
query_candidates = [
    c for c in df.columns
    if c.lower().strip().replace('_', ' ') in [
        'query', 'text', 'comment', 'tweet', 'sentence', 'content', 'message'
    ]
]

# Select the first matching query column, or fall back to the first object/string column.
query_col = query_candidates[0] if query_candidates else df.select_dtypes(include='object').columns[0]

# Find the image-description column because this dataset uses BOTH text fields to determine the label.
image_candidates = [
    c for c in df.columns
    if c.lower().strip().replace('_', ' ') in [
        'image description', 'image_description', 'image desc', 'image text',
        'image caption', 'caption', 'description'
    ]
    and c != query_col
]

# Select the first matching image-description column.
if image_candidates:
    image_col = image_candidates[0]
else:
    # Try a broader search for a column whose name contains both "image" and "description".
    image_candidates = [
        c for c in df.columns
        if 'image' in c.lower() and ('description' in c.lower() or 'desc' in c.lower())
        and c != query_col
    ]
    if not image_candidates:
        raise ValueError(
            'Could not find the image-description column. '
            f'Available columns are: {list(df.columns)}'
        )
    image_col = image_candidates[0]

# Find the target/label column while avoiding both input columns.
label_candidates = [
    c for c in df.columns
    if c not in [query_col, image_col]
    and c.lower().strip() in ['label', 'target', 'class', 'category', 'toxic', 'y']
]

# Use a standard label column when one exists.
if label_candidates:
    label_col = label_candidates[0]
else:
    # Otherwise choose the last non-input column as the target.
    non_input_cols = [c for c in df.columns if c not in [query_col, image_col]]
    label_col = non_input_cols[-1]

print('Query column:', query_col)
print('Image-description column:', image_col)
print('Label column:', label_col)
print('\nClass distribution:')
print(df[label_col].value_counts())


## 3. Remove duplicates and missing values

**Important dataset rule:** the label belongs to the **query + image description pair**, not to the query by itself.

Therefore, the same query is allowed to appear with different image descriptions and different labels. We must **not** remove those rows.

We remove:
1. Rows where the query, image description, or label is missing.
2. True exact duplicates where **query + image description + label** are identical.

We do **not** remove a repeated query just because its label is different.


In [ ]:
print('Rows before cleaning:', len(df))

# Remove rows where any part of the complete training example is missing.
before = len(df)
df = df.dropna(subset=[query_col, image_col, label_col]).copy()
print(f'Removed {before - len(df)} rows with missing query/image-description/label.')

# Convert input columns to strings so text processing is consistent.
df[query_col] = df[query_col].astype(str).str.strip()
df[image_col] = df[image_col].astype(str).str.strip()

# Strip stray whitespace from labels so "Safe " and "Safe" are treated as the same class.
df[label_col] = df[label_col].astype(str).str.strip()

# Remove ONLY true duplicate examples.
# The complete pair is query + image description + label.
before = len(df)
df = df.drop_duplicates(
    subset=[query_col, image_col, label_col]
).reset_index(drop=True)
print(
    f'Removed {before - len(df)} true exact duplicates '
    f'(same query + image description + label).'
)

# Show repeated queries with different labels as a diagnostic, NOT as rows to delete.
# This confirms that the dataset can legitimately contain the same query more than once.
query_label_counts = (
    df.groupby(query_col)[label_col]
      .nunique()
)
repeated_query_count = int((query_label_counts > 1).sum())

print(
    f'Queries appearing with multiple labels: {repeated_query_count}. '
    'These rows are kept because the image description is part of the input.'
)

print('Rows after cleaning:', len(df))
print('\nClass distribution after cleaning:')
print(df[label_col].value_counts())


In [ ]:
# Show a few repeated queries to verify that we are keeping different image-description pairs.
repeated_queries = (
    df.groupby(query_col)
      .filter(lambda group: len(group) > 1)
      [[query_col, image_col, label_col]]
)

print('Number of rows belonging to repeated queries:', len(repeated_queries))

if len(repeated_queries) > 0:
    print(
        repeated_queries
        .head(10)
        .to_string(index=False)
    )
else:
    print('No repeated queries were found in the cleaned dataset.')


### 3b. Drop classes too small to stratify-split

A stratified 60/20/20 train/val/test split needs at least a few examples per class (enough that `temp` still gets at least 2, so it can itself be split into val and test). The duplicate/conflicting-label removal above can shrink an already-rare class down to just a handful of rows. Any class below `MIN_CLASS_COUNT` is dropped here (and reported), rather than crashing `train_test_split` later.

In [ ]:
MIN_CLASS_COUNT = 10  # Keep classes with enough examples for a stratified train/validation/test split.

class_counts_now = df[label_col].value_counts()
print('Class counts before rare-class filtering:')
print(class_counts_now)
# A class needs multiple examples in order to appear in the stratified splits.
rare_classes = class_counts_now[class_counts_now < MIN_CLASS_COUNT].index.tolist()

if rare_classes:
    print(
        f'\nDropping {len(rare_classes)} class(es) with fewer than '
        f'{MIN_CLASS_COUNT} rows: {rare_classes}'
    )
    before = len(df)
    df = df[~df[label_col].isin(rare_classes)].reset_index(drop=True)
    print(f'Removed {before - len(df)} rows belonging to those classes.')
else:
    print('\nNo classes are below the minimum count - nothing dropped.')

print('\nFinal class distribution going into tokenization/splitting:')
print(df[label_col].value_counts())


## 4. Combine the query + image description, then sentence- and word-tokenize

The model must receive **both input fields** because the target label depends on their combination.

We join them with a special `SEP` token:

`query + SEP + image description`

Then we:
1. Normalize the combined text.
2. Split it into sentences with NLTK.
3. Split each sentence into word tokens.
4. Encode the labels as integer IDs.


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('stopwords')  # Download the English stopword list.
nltk.download('punkt')  # Download the sentence/word tokenizer data.
nltk.download('punkt_tab')  # Required by newer NLTK versions.

stop_words = set(stopwords.words('english'))  # Keep the list available for optional future filtering.

def clean_text(text):
    text = str(text).lower()  # Convert the input text to lowercase.
    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)  # Replace web links with a common token.
    text = re.sub(r'[^a-z0-9\s!?.]', ' ', text)  # Remove symbols while keeping basic sentence punctuation.
    text = re.sub(r'\s+', ' ', text).strip()  # Collapse repeated whitespace.
    return text  # Return the cleaned text.

def combine_inputs(row):
    query = clean_text(row[query_col])  # Clean the query text.
    image_description = clean_text(row[image_col])  # Clean the image description.
    return f'{query} SEP {image_description}'  # Keep both fields and mark their boundary.

def tokenize_text(text):
    sentences = sent_tokenize(text)  # Split the combined input into sentences.
    tokens = []  # Create a list that will contain all word tokens.
    for sentence in sentences:  # Process each sentence separately.
        tokens.extend(word_tokenize(sentence))  # Split the sentence into word-level tokens.
    return tokens, len(sentences)  # Return tokens and the number of sentences.

# Combine the two input columns before tokenization.
df['combined_text'] = df.apply(combine_inputs, axis=1)

# Apply sentence + word tokenization to the combined input.
tokenized = df['combined_text'].apply(tokenize_text)

df['tokens'] = tokenized.apply(lambda t: t[0])  # Store the word tokens for each example.
df['num_sentences'] = tokenized.apply(lambda t: t[1])  # Store the sentence count for each example.

label_encoder = LabelEncoder()  # Create an encoder that maps class names to integer IDs.
df['label_id'] = label_encoder.fit_transform(df[label_col])  # Convert labels into 0, 1, 2, ...

print('Classes:', list(label_encoder.classes_))
print('Average sentences per example:', round(df['num_sentences'].mean(), 2))
print('Average word tokens per example:', round(df['tokens'].apply(len).mean(), 2))
print(df[[query_col, image_col, 'combined_text', 'tokens', 'label_id']].head())


## 5. Split the dataset

We use a stratified split so that class proportions remain approximately similar across training, validation, and test sets.

The split is performed **after the two input columns have been combined**, but before oversampling.

Oversampling is applied to the **training set only**. Validation and test remain untouched so their scores reflect the original data distribution.


In [ ]:
tokens_list = df['tokens'].tolist()  # Get the token list for every combined query + image example.
labels = df['label_id'].to_numpy()  # Get the numeric label for every example.

train_tokens, temp_tokens, train_labels, temp_labels = train_test_split(
    tokens_list,
    labels,
    test_size=0.40,
    random_state=SEED,
    stratify=labels
)  # Use 60% for training and 40% temporarily for validation/test.

val_tokens, test_tokens, val_labels, test_labels = train_test_split(
    temp_tokens,
    temp_labels,
    test_size=0.25,
    random_state=SEED,
    stratify=temp_labels
)  # Split the remaining 20% equally into validation and test.

print('Training samples:', len(train_tokens))
print('Validation samples:', len(val_tokens))
print('Test samples:', len(test_tokens))


## 6. Handle class imbalance with random oversampling

Class imbalance is handled on the **training set only**.

We oversample minority classes with replacement so they receive more training examples. Validation and test are never oversampled.

The earlier bug was not oversampling itself; it was that the dataset had already been severely reduced by incorrectly deleting valid repeated-query examples.


In [ ]:
from collections import Counter  # Used to count examples in each class.

class_counts = Counter(train_labels)  # Count the training examples for every class.

print('Training class counts before oversampling:')
for cls, count in sorted(class_counts.items()):
    print(f'  {label_encoder.classes_[cls]}: {count}')

max_count = max(class_counts.values())  # Use the largest class as the target size.
rng = np.random.default_rng(SEED)  # Create a reproducible random generator.

balanced_tokens = []  # Store the oversampled token sequences.
balanced_labels = []  # Store the corresponding labels.

for cls, count in sorted(class_counts.items()):
    idx = np.where(train_labels == cls)[0]  # Find all training indices for this class.

    if count < max_count:
        extra_idx = rng.choice(
            idx,
            size=max_count - count,
            replace=True
        )  # Randomly duplicate minority examples until they match the largest class.
        idx = np.concatenate([idx, extra_idx])  # Add the sampled examples to the original indices.

    balanced_tokens.extend([train_tokens[i] for i in idx])  # Add the selected token sequences.
    balanced_labels.extend([cls] * len(idx))  # Add the matching labels.

# Shuffle the oversampled training data so duplicate examples are distributed across batches.
shuffle_idx = rng.permutation(len(balanced_labels))
train_tokens = [balanced_tokens[i] for i in shuffle_idx]
train_labels = np.array(balanced_labels)[shuffle_idx]

print('\nTraining class counts after oversampling:')
for cls, count in sorted(Counter(train_labels).items()):
    print(f'  {label_encoder.classes_[cls]}: {count}')

print('\nTotal training samples after oversampling:', len(train_labels))


## 7. Build a vocabulary and convert tokens into sequences

An RNN cannot directly process raw words.

We create a vocabulary using the **training data only**, then replace each token with an integer ID.

The special `SEP` token is kept in the vocabulary so the model can learn where the query ends and the image description begins.


In [ ]:
PAD_TOKEN = '<PAD>'  # Special token used to pad shorter sequences.
UNK_TOKEN = '<UNK>'  # Special token used for words not seen during training.

counter = Counter()  # Create a word-frequency counter.

for tokens in train_tokens:
    counter.update(tokens)  # Count every token in the training data.

MIN_FREQ = 1  # Keep words that appear at least once in training.
MAX_VOCAB_SIZE = 30000  # Limit vocabulary size to control memory usage.

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}  # Reserve IDs for padding and unknown words.

for word, freq in counter.most_common(MAX_VOCAB_SIZE):
    if freq >= MIN_FREQ and word not in vocab:
        vocab[word] = len(vocab)  # Assign the next integer ID to this word.

MAX_LEN = 160  # Allow room for both the query and image description.

def encode_tokens(tokens):
    tokens = tokens[:MAX_LEN]  # Truncate very long combined examples.
    ids = [
        vocab.get(token, vocab[UNK_TOKEN])
        for token in tokens
    ]  # Convert every token to its vocabulary ID.
    ids += [
        vocab[PAD_TOKEN]
    ] * (MAX_LEN - len(ids))  # Pad short examples to the fixed length.
    return ids  # Return the fixed-length sequence.

X_train = np.array(
    [encode_tokens(tokens) for tokens in train_tokens],
    dtype=np.int64
)  # Encode training examples.

X_val = np.array(
    [encode_tokens(tokens) for tokens in val_tokens],
    dtype=np.int64
)  # Encode validation examples.

X_test = np.array(
    [encode_tokens(tokens) for tokens in test_tokens],
    dtype=np.int64
)  # Encode test examples.

print('Vocabulary size:', len(vocab))
print('Sequence shape:', X_train.shape)


## 8. Create the PyTorch Dataset and DataLoaders

A Dataset stores the encoded combined query + image-description sequences and labels.

A DataLoader groups examples into mini-batches for training and evaluation.


In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)  # Store token IDs as integer tensors.
        self.y = torch.tensor(y, dtype=torch.long)  # Store labels as integer tensors.

    def __len__(self):
        return len(self.y)  # Return the number of examples.

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]  # Return one input sequence and its label.

BATCH_SIZE = 64  # Number of examples processed in one training step.

train_loader = DataLoader(
    TextDataset(X_train, train_labels),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)  # Create shuffled training batches.

val_loader = DataLoader(
    TextDataset(X_val, val_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)  # Create validation batches without shuffling.

test_loader = DataLoader(
    TextDataset(X_test, test_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)  # Create test batches without shuffling.


## 9. Define the RNN model

The model has:
1. An embedding layer that converts word IDs into dense vectors.
2. A bidirectional RNN that learns context from the combined query + image description.
3. Masked mean pooling so padding tokens do not become the final representation.
4. A dropout layer for regularization.
5. A linear classifier that produces one score for every class.

Using masked pooling is important because the old `output[:, -1, :]` used the **last padded position** for shorter sequences, which can throw away useful information.


In [ ]:
class RNNModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()  # Initialize the PyTorch module.

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )  # Convert token IDs into dense vectors.

        self.sequence_model = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )  # Process the complete combined text in both directions.

        self.dropout = nn.Dropout(dropout)  # Randomly drop some values during training.

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )  # Convert the bidirectional representation into class logits.

    def forward(self, x):
        mask = (x != 0).float()  # Mark real tokens as 1 and padding tokens as 0.

        embedded = self.embedding(x)  # Convert token IDs into embedding vectors.

        output, _ = self.sequence_model(embedded)  # Run the sequence through the bidirectional RNN.

        mask = mask.unsqueeze(-1)  # Change mask shape so it can multiply the RNN output.

        masked_output = output * mask  # Remove padding positions from the representation.

        token_counts = mask.sum(dim=1).clamp(min=1.0)  # Count real tokens in each example.

        representation = masked_output.sum(dim=1) / token_counts  # Compute the mean over real tokens only.

        representation = self.dropout(representation)  # Apply dropout before classification.

        logits = self.classifier(representation)  # Produce one score for every class.

        return logits  # Return the raw class scores.

NUM_CLASSES = len(label_encoder.classes_)  # Count the target classes.

model = RNNModel(
    len(vocab),
    128,
    128,
    NUM_CLASSES
).to(device)  # Create the model and move it to CPU or GPU.

print(model)  # Display the model architecture.


## 10. Training setup

We use cross-entropy loss for multi-class classification and AdamW for optimization. The learning-rate scheduler reduces the learning rate when validation F1 stops improving.

In [ ]:
criterion = nn.CrossEntropyLoss()  # Measure how different the predicted class scores are from the true labels.
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)  # Update model weights using AdamW.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=1)  # Lower LR if validation F1 stalls.

EPOCHS = 10  # Maximum number of complete passes through the training data.
best_f1 = -1.0  # Store the best validation F1 observed so far.
best_state = None  # This variable will hold the best model parameters.

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}  # Store metrics for graphs.


## 11. Define the evaluation function

F1 is the required evaluation metric. We use **macro F1**, which calculates F1 for each class and gives each class equal importance.

In [ ]:
def evaluate(model, loader):  # Evaluate the model on a complete DataLoader.
    model.eval()  # Put the model into evaluation mode; dropout is disabled.
    total_loss = 0.0  # Accumulate validation/test loss.
    all_preds = []  # Store all predicted labels.
    all_targets = []  # Store all true labels.

    with torch.no_grad():  # Disable gradient calculation because we are not training here.
        for inputs, targets in loader:  # Process each batch.
            inputs = inputs.to(device, non_blocking=True)  # Move input data to GPU/CPU.
            targets = targets.to(device, non_blocking=True)  # Move labels to GPU/CPU.
            logits = model(inputs)  # Generate class scores.
            loss = criterion(logits, targets)  # Calculate the batch loss.
            total_loss += loss.item() * inputs.size(0)  # Add the batch's total loss contribution.
            preds = torch.argmax(logits, dim=1)  # Select the class with the highest score.
            all_preds.extend(preds.cpu().numpy())  # Move predictions back to CPU and store them.
            all_targets.extend(targets.cpu().numpy())  # Move true labels back to CPU and store them.

    avg_loss = total_loss / len(loader.dataset)  # Calculate average loss over all examples.
    f1 = f1_score(all_targets, all_preds, average='macro')  # Calculate macro F1 score.
    return avg_loss, f1, np.array(all_targets), np.array(all_preds)  # Return all evaluation information.


## 12. Train the model

The best model is selected according to validation macro F1, not accuracy.

In [ ]:
for epoch in range(1, EPOCHS + 1):  # Repeat the training process for each epoch.
    model.train()  # Put the model into training mode.
    running_loss = 0.0  # Reset the accumulated training loss.
    train_preds = []  # Store predictions from the training epoch.
    train_targets = []  # Store true labels from the training epoch.

    for inputs, targets in train_loader:  # Loop through every training batch.
        inputs = inputs.to(device, non_blocking=True)  # Move input tensors to the selected device.
        targets = targets.to(device, non_blocking=True)  # Move target tensors to the selected device.
        optimizer.zero_grad(set_to_none=True)  # Clear gradients from the previous batch.
        logits = model(inputs)  # Run a forward pass through the model.
        loss = criterion(logits, targets)  # Calculate the classification loss.
        loss.backward()  # Calculate gradients using backpropagation.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Prevent extremely large gradients.
        optimizer.step()  # Update the model weights using the calculated gradients.
        running_loss += loss.item() * inputs.size(0)  # Add this batch's loss to the epoch total.
        preds = torch.argmax(logits, dim=1)  # Select the predicted class for each example.
        train_preds.extend(preds.detach().cpu().numpy())  # Store predictions on the CPU.
        train_targets.extend(targets.detach().cpu().numpy())  # Store true labels on the CPU.

    train_loss = running_loss / len(train_loader.dataset)  # Calculate average training loss.
    train_f1 = f1_score(train_targets, train_preds, average='macro')  # Calculate training macro F1.
    val_loss, val_f1, _, _ = evaluate(model, val_loader)  # Evaluate on validation data.
    scheduler.step(val_f1)  # Tell the scheduler whether validation F1 improved.

    history['train_loss'].append(train_loss)  # Save training loss for the graph.
    history['val_loss'].append(val_loss)  # Save validation loss for the graph.
    history['train_f1'].append(train_f1)  # Save training F1 for the graph.
    history['val_f1'].append(val_f1)  # Save validation F1 for the graph.

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}')  # Show progress.

    if val_f1 > best_f1:  # Check whether this is the best validation F1 so far.
        best_f1 = val_f1  # Update the best validation F1.
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}  # Save a CPU copy of the best weights.

model.load_state_dict(best_state)  # Restore the model that achieved the best validation F1.


## 13. Plot training results

These graphs show how loss and macro F1 changed during training.

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)  # Create x-axis values for each epoch.
model_name = "RNN"
plt.figure(figsize=(8, 5))  # Create a figure for the loss curves.
plt.plot(epochs, history['train_loss'], label='Train Loss')  # Plot training loss.
plt.plot(epochs, history['val_loss'], label='Validation Loss')  # Plot validation loss.
plt.xlabel('Epoch')  # Label the x-axis.
plt.ylabel('Loss')  # Label the y-axis.
plt.title(f'{model_name} Training and Validation Loss')  # Add a graph title.
plt.legend()  # Display the curve labels.
plt.grid(True)  # Add a light grid for easier reading.
plt.show()  # Display the loss graph.

plt.figure(figsize=(8, 5))  # Create a separate figure for F1 curves.
plt.plot(epochs, history['train_f1'], label='Train Macro F1')  # Plot training F1.
plt.plot(epochs, history['val_f1'], label='Validation Macro F1')  # Plot validation F1.
plt.xlabel('Epoch')  # Label the x-axis.
plt.ylabel('Macro F1')  # Label the y-axis.
plt.title(f'{model_name} Training and Validation Macro F1')  # Add a graph title.
plt.legend()  # Display curve labels.
plt.grid(True)  # Add a grid to make the graph easier to read.
plt.show()  # Display the F1 graph.


## 14. Final test evaluation and confusion matrix

The test set is evaluated only after training is complete.

In [ ]:
#test_loss, test_f1, y_true, y_pred = evaluate(model, test_loader)
# Evaluate the final selected model on the unseen test data.
#print(f"Final Test Loss: {test_loss:.4f}")
# Print the final test loss.
#print(f"Final Test Macro F1: {test_f1:.4f} ({test_f1 * 100:.2f}%)")
# Print the final Macro F1 score as a percentage.
#print("\nClassification Report:")
# Add a heading for the detailed classification report.
# Create a list containing every class index known to the label encoder.
#all_labels = np.arange(len(label_encoder.classes_))
#print(
   # classification_report(
       # y_true,
        #y_pred,
        #labels=all_labels,
        #target_names=label_encoder.classes_,
        #zero_division=0
  #  )
#)
# 'labels' tells sklearn to include all 9 classes.
# 'target_names' provides the names for those 9 classes.
# If a class has no samples in y_true, its metrics will be shown as 0.
# Calculate the confusion matrix using all classes.
#cm = confusion_matrix(
   # y_true,
   # y_pred,
   # labels=all_labels
#)
# Create the confusion-matrix display.
#disp = ConfusionMatrixDisplay(
  #  confusion_matrix=cm,
  #  display_labels=label_encoder.classes_
#)

# Create the figure.
#fig, ax = plt.subplots(figsize=(9, 7))


# Draw the confusion matrix.
#disp.plot(
#    ax=ax,
#    xticks_rotation=45,
#    cmap="Blues",
#    colorbar=False
#)


# Add a title.
#plt.title(f"{model_name} Confusion Matrix")


# Prevent labels from being cut off.
#plt.tight_layout()


# Display the confusion matrix.
#plt.show()

In [ ]:
test_loss, test_f1, y_true, y_pred = evaluate(model, test_loader)  # Evaluate the final selected model on unseen test data.
print(f'Final Test Loss: {test_loss:.4f}')  # Print test loss.
print(f'Final Test Macro F1: {test_f1:.4f} ({test_f1 * 100:.2f}%)')  # Print the required F1 metric as a percentage.
print('\nClassification Report:')  # Add a heading for the detailed report.
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, zero_division=0))  # Print precision, recall and F1 for every class.

cm = confusion_matrix(y_true, y_pred)  # Calculate the confusion matrix.
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)  # Create a visual confusion-matrix object.
fig, ax = plt.subplots(figsize=(7, 6))  # Create a figure for the confusion matrix.
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)  # Draw the confusion matrix.
plt.title(f'{model_name} Confusion Matrix')  # Add a title.
plt.tight_layout()  # Prevent labels from being cut off.
plt.show()  # Display the confusion matrix.


## 15. Save the trained model

The weights, vocabulary, class names, and sequence length are saved so the model can be loaded later without training from scratch.

The saved vocabulary includes the `SEP` token used to separate the query from the image description.


In [ ]:
MODEL_FILE = 'rnn_model_improved_pair_input.pt'  # Save the corrected pair-input model separately.

torch.save({
    'model_state_dict': model.state_dict(),  # Save the learned neural-network parameters.
    'vocab': vocab,  # Save the vocabulary needed to encode future examples.
    'classes': label_encoder.classes_.tolist(),  # Save the original class names.
    'max_len': MAX_LEN,  # Save the sequence length used by the model.
    'query_column': query_col,  # Save the query column name.
    'image_column': image_col,  # Save the image-description column name.
}, MODEL_FILE)  # Write all saved information to disk.

print('Saved model to:', MODEL_FILE)  # Confirm the saved model path.
